# Exploring Symbolic Regression with PySR

Context: In our consulting project, we're working with the AlphaPEM model of a PEM fuel cell. For each simulation run, we vary operating conditions and some physical parameters, and the model gives us a polarization curve: voltage values for a range of current densities.

Our goal is to build a symbolic surrogate model: a simple, interpretable equation that predicts the voltage at each current density from the input parameters.

PySR (Symbolic Regression in Python) could help with this by finding analytical expressions that describe the relationship between the input parameters and the voltage. In this notebook, we're testing PySR on a reduced dataset to see how well it works and whether this approach is promising before applying it to the full AlphaPEM data.



In [1]:
!pip install -U pysr

In [3]:
!pip install xgboost

   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
    --------------------------------------- 2.9/150.0 MB 15.2 MB/s eta 0:00:10
   - -------------------------------------- 6.6/150.0 MB 16.1 MB/s eta 0:00:09
   -- ------------------------------------- 9.7/150.0 MB 15.5 MB/s eta 0:00:10
   --- ------------------------------------ 12.3/150.0 MB 14.9 MB/s eta 0:00:10
   ---- ----------------------------------- 15.2/150.0 MB 14.5 MB/s eta 0:00:10
   ---- ----------------------------------- 18.4/150.0 MB 14.5 MB/s eta 0:00:10
   ----- ---------------------------------- 21.5/150.0 MB 14.5 MB/s eta 0:00:09
   ------ --------------------------------- 24.6/150.0 MB 14.6 MB/s eta 0:00:09
   ------- -------------------------------- 27.5/150.0 MB 14.4 MB/s eta 0:00:09
   -------- ------------------------------- 30.7/150.0 MB 14.5 MB/s eta 0:00:09
   -------- ------------------------------- 33.6/150.0 MB 14.4 MB/s eta 0:00:09
   --------- ------------------------------ 36.2/150

In [4]:
import sympy
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

## Load the data
We have a CSV file with simulation results: each row corresponds to one configuration (a combination of operating conditions and physical parameters), and contains voltage and current density values along the polarization curve.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

# Define the path to the CSV file in my Drive
csv_path = '/content/drive/MyDrive/Colab Notebooks/validated_sobol_samples.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
csv_path = r"../data/processed/validated_sobol_samples.csv"

In [7]:
data = pd.read_csv(csv_path)
print("There are", len(data), "polarization curves available.")

simplified = True
if simplified:
  data = data.sample(n=500, random_state=0)
  print("For test mode, only using", len(data), "of them.")

data.head()

There are 24784 polarization curves available.
For test mode, only using 500 of them.


,Tfc,Pa_des,Sc,Phi_c_des,epsilon_gdl,tau,epsilon_mc,epsilon_c,e,Re,...,Ucell_22,Ucell_23,Ucell_24,Ucell_25,Ucell_26,Ucell_27,Ucell_28,Ucell_29,Ucell_30,Ucell_31
4129,355.616040,139479.525101,2.171168,0.281209,0.793171,1.166688,0.203111,0.159460,5,1.573953e-06,...,0.566224,0.554381,0.542675,0.531089,0.519607,0.508215,0.496899,0.485647,0.474448,0.463290
15081,346.731756,223681.368397,2.839130,0.444682,0.639230,1.182556,0.217391,0.164810,3,4.559323e-06,...,0.579767,0.565365,0.551070,0.536869,0.522752,0.508705,0.494719,0.480786,0.466895,0.453038
16547,347.122538,176886.567664,1.523028,0.682170,0.594395,1.791487,0.388760,0.220939,3,7.135004e-07,...,0.555722,0.545316,0.534999,0.524756,0.514569,0.504424,0.494305,0.484196,0.474082,0.463945
4273,357.759896,174865.041329,2.542421,0.398989,0.789934,3.431252,0.151824,0.207358,5,4.527742e-06,...,0.358694,0.341403,0.324478,0.307895,0.291633,0.275673,0.259999,0.244593,0.229441,0.214530
4254,346.077803,145209.330349,1.367694,0.353185,0.609637,2.927453,0.311690,0.258568,5,1.101275e-06,...,0.578758,0.569778,0.560319,0.550407,0.540042,0.529281,0.518064,0.506486,0.494582,0.482292


We define which parameters we used as inputs when generating the simulations. These will be used as features for the surrogate model.

In [8]:
PARAMETER_RANGES = {
    'Tfc': [333, 363],
    'Pa_des': [130000.0, 300000.0],
    'Sc': [1.1, 3.0],
    'Phi_c_des': [0.1, 0.7],
    'epsilon_gdl': [0.55, 0.8],
    'tau': [1.0, 4.0],
    'epsilon_mc': [0.15, 0.4],
    'epsilon_c': [0.15, 0.3],
    'e': [3, 4, 5],
    'Re': [5e-07, 5e-06],
    'i0_c_ref': [0.001, 80],
    'kappa_co': [15, 40],
    'kappa_c': [0, 2.5],
}
INPUT_COLUMNS = list(PARAMETER_RANGES.keys())

## Convert to long format for symbolic regression

Each simulation gives us a full polarization curve, i.e., multiple current and voltage values per configuration. To apply symbolic regression, we want to work at the level of individual (input parameters, current) → voltage data points.

We melt the DataFrame so each row corresponds to one point on a polarization curve.

In [9]:
def melt_polarization_data(df, input_cols, current_cols_prefix="ifc_", voltage_cols_prefix="Ucell_"):
    n_points = sum(col.startswith(current_cols_prefix) for col in df.columns)
    long_df_list = []

    for i in range(n_points):
        current_col = f"{current_cols_prefix}{i+1}"
        voltage_col = f"{voltage_cols_prefix}{i+1}"

        temp = df[input_cols].copy()
        temp["current"] = df[current_col]
        temp["voltage"] = df[voltage_col]
        long_df_list.append(temp)

    return pd.concat(long_df_list, ignore_index=True)

# Apply transformation
long_df = melt_polarization_data(data, INPUT_COLUMNS)
print("Long DataFrame shape:", long_df.shape)
long_df.head()


Long DataFrame shape: (15500, 15)


,Tfc,Pa_des,Sc,Phi_c_des,epsilon_gdl,tau,epsilon_mc,epsilon_c,e,Re,i0_c_ref,kappa_co,kappa_c,current,voltage
0,355.616040,139479.525101,2.171168,0.281209,0.793171,1.166688,0.203111,0.159460,5,1.573953e-06,30.769340,38.736597,1.427681,0.000807,0.987195
1,346.731756,223681.368397,2.839130,0.444682,0.639230,1.182556,0.217391,0.164810,3,4.559323e-06,32.798461,38.987409,1.280134,0.000819,1.003825
2,347.122538,176886.567664,1.523028,0.682170,0.594395,1.791487,0.388760,0.220939,3,7.135004e-07,11.809619,39.431162,0.522485,0.000806,0.892559
3,357.759896,174865.041329,2.542421,0.398989,0.789934,3.431252,0.151824,0.207358,5,4.527742e-06,51.556081,34.508391,0.322468,0.000815,0.960595
4,346.077803,145209.330349,1.367694,0.353185,0.609637,2.927453,0.311690,0.258568,5,1.101275e-06,10.652567,21.001926,0.560411,0.000816,0.952009


## Fit a symbolic model to AlphaPEM outputs

Now that the data is in long format (long_df), we’ll use it to learn an equation:
(operating conditions, physical parameters, current) → voltage

Do we need a train/test split here?

I think yes. Because we're working with "noisy" simulation data (not a known function), we should hold out part of the data to test generalization. This helps us avoid overfitting and check whether the learned equation captures general patterns rather than just memorizing the data.

In [10]:
# Define input and target columns
feature_cols = INPUT_COLUMNS + ["current"]
target_col = "voltage"

# Extract X and y
X = long_df[feature_cols]
y = long_df[target_col]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (9300, 14)
X_test shape: (6200, 14)
y_train shape: (9300,)
y_test shape: (6200,)


In [ ]:
model = PySRRegressor(
    niterations=100,
    populations=30,
    model_selection="best",
    binary_operators=["+", "*", "-", "/"],
    unary_operators=["cos", "sin", "exp", "log"],
    select_k_features=5,
    loss="loss(x, y) = (x - y)^2",
)

model.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/pysr/sr.py:1036: FutureWarning: `loss` has been renamed to `elementwise_loss` in PySRRegressor. Please use that instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Using features ['Phi_c_des' 'tau' 'epsilon_c' 'kappa_c' 'current']
INFO:pysr.sr:Using features ['Phi_c_des' 'tau' 'epsilon_c' 'kappa_c' 'current']
[ Info: Started!


Se truncaron las últimas líneas 5000 del resultado de transmisión.
21          4.280e-01  1.168e-02  y = sin(exp(-0.27352 / sin(cos((sin(cos(exp(exp(tau)))) * ...
                                      (0.87113 - (tau + 0.071753))) * -0.64251)))) - -0.075802
23          4.202e-01  9.166e-03  y = sin(exp(0.48148 / sin(cos(sin(cos(exp(exp(tau)))) * ((...
                                      tau - sin(sin(exp(tau)))) * -0.3123))))) - sin(0.32269)
24          4.154e-01  1.149e-02  y = sin(exp(0.48148 / sin(cos(-0.3123 * (sin(cos(exp(exp(t...
                                      au)))) * ((sin(exp(tau)) - tau) - 0.043279)))))) - sin(0.3...
                                      2269)
26          4.097e-01  7.006e-03  y = (sin(exp(0.36728 / sin(cos((sin(cos(exp(exp(tau)))) * ...
                                      (-0.051295 - (sin(sin(exp(tau))) - tau))) * -0.35068)))) -...
                                       0.3864) + 0.066697
──────────────────────────────────────────────────────────

We can print the model, which will print out all the discovered expressions:

In [ ]:
model

We can also view the SymPy format of the best expression:

In [ ]:
model.sympy()

Let's check the performance of the symbolic model we trained.

In [ ]:
# Predict with symbolic regression model
y_pred_train_sym = model.predict(X_train)
y_pred_test_sym = model.predict(X_test)

# MSE
mse_train_sym = mean_squared_error(y_train, y_pred_train_sym)
mse_test_sym = mean_squared_error(y_test, y_pred_test_sym)

print(f"Symbolic Regression Train MSE: {mse_train_sym:.5f}")
print(f"Symbolic Regression Test MSE:  {mse_test_sym:.5f}")

### Tuning PySR for better symbolic performance

We'll increase the search effort, enable feature selection, and replace less relevant operators with `inv` (unary) and `/` (binary) to better capture inverse relationships we expect in polarization curves.

Inspiration: [Implementing mathematical constraints using PySR](https://github.com/MilesCranmer/PySR/discussions/667)



In [ ]:
custom_monotonic_loss = """
function eval_loss(tree, dataset::Dataset{T,L}, options)::L where {T,L}
    # Index of current in input matrix (last column)
    derivative_with_respect_to = size(dataset.X, 2)

    # Evaluate prediction and its derivative w.r.t. current
    prediction, gradient, complete = eval_diff_tree_array(tree, dataset.X, derivative_with_respect_to, options)

    if !complete
        return L(Inf)
    end

    # Penalize positive gradients: we expect dV/dI <= 0
    monotonicity_penalty = sum(i -> gradient[i] > 0 ? abs2(gradient[i]) : L(0), eachindex(gradient))

    # Mean squared error loss
    mse = sum((prediction .- dataset.y).^2)

    # Combine loss with penalty
    beta = L(1e-3)  # <-- tunable parameter
    return (mse + beta * monotonicity_penalty) / dataset.n
end
"""

In [ ]:
model_imp = PySRRegressor(
    niterations=100,
    populations=50,
    model_selection="best",
    select_k_features=5,
    binary_operators=["+", "-", "*", "/", "pow"],
    unary_operators=["inv", "log", "exp"],  # inverse-focused
    extra_sympy_mappings={'inv': lambda x: 1/x},
)

model_imp.fit(X_train, y_train)
print(model_imp)

## Compare with a Black-Box Model (XGBoost)

Now that we've trained a symbolic model, we want to compare it against a non-interpretable machine learning model. The goal is to understand how much accuracy we might be giving up for the sake of interpretability.

We'll train an XGBoost regressor on the same input features (INPUT_COLUMNS + ["current"]) to predict voltage, and evaluate its performance on both the training and test sets. This gives us a reference point for how good a purely predictive model can get.

Then we'll compare its mean squared error (MSE) to the one from the symbolic regression.

In [39]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
# Train XGBoost on the same data
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=0,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

# Predict
y_pred_train_xgb = xgb_model.predict(X_train)
y_pred_test_xgb = xgb_model.predict(X_test)

# MSE
mse_train_xgb = mean_squared_error(y_train, y_pred_train_xgb)
mse_test_xgb = mean_squared_error(y_test, y_pred_test_xgb)

print(f"XGBoost Train MSE: {mse_train_xgb:.5f}")
print(f"XGBoost Test MSE:  {mse_test_xgb:.5f}")


XGBoost Train MSE: 0.00643
XGBoost Test MSE:  0.01784
